# RegressLM — Table 3 reproduction (full scale)

Reproduces the three official claims for *Regression Language Models for Code* (OpenReview `utTapVWtc7`):
1. A single RLM predicts memory/latency/accuracy of code across multiple languages.
2. 300M RLM (T5Gemma init) obtains **>0.9 Spearman on APPS** (paper Table 3 = 0.930; dataset card reports **0.926**).
3. Unified model obtains **>0.5 average Spearman across 17 CodeNet languages**.

Uses the released checkpoint `akhauriyash/RegressLM-gemma-s-RLM-table3` and dataset
`akhauriyash/Code-Regression`, with the authors' published eval recipe (dataset card):
per-space input prefix, `generate(do_sample=True, top_p=0.95, temperature=1.0,
min=max_new_tokens=N_OUT)`, `token_ids_to_floats`[0], median over 8 samples, Spearman.

Mirrors `repro/src/run_eval.py`. Run on a **GPU runtime** (T4/L4/A100).

In [ ]:
import os, math, sys, subprocess
# CRITICAL: pin transformers==4.53.2 (checkpoint export version). regress-lm[extras]
# pulls transformers>5.0.0, but 5.x breaks this T5Gemma model's generate() (encoder
# signal never reaches decoder -> constant output). Install LAST to override.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'regress-lm[extras]', 'pyarrow', 'datasets', 'scipy', 'numpy', 'pandas'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.53.2'])
import torch, numpy as np
from scipy import stats
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pyarrow as pa, pyarrow.dataset as ds
from ast import literal_eval
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device=='cuda' else torch.float32
print('device', device, '| transformers', __import__('transformers').__version__,
      '| GPU', torch.cuda.get_device_name(0) if device=='cuda' else 'none')

In [ ]:
import os, shutil, glob, urllib.request
from huggingface_hub import snapshot_download
REPO='akhauriyash/RegressLM-gemma-s-RLM-table3'
# snapshot_download (public repo, no token needed) -> local dir so the bundled
# encoder_tokenizer/ is used (avoids gated google/t5gemma for the encoder vocab).
CKPT_DIR = snapshot_download(REPO)
PARQUET='hf://datasets/akhauriyash/Code-Regression/data.parquet'

# Apply repro patches to the downloaded code BEFORE loading:
#  - configuration_regresslm.py: skip the gated google/t5gemma backbone auto-download
#    (raised GatedRepoError 401 on the throwaway default config instance).
#  - modeling_regresslm.py: tie_weights(**kwargs) for newer-transformers compat.
PATCH_RAW='https://raw.githubusercontent.com/MachineLearning-Nerd/icml26-repro-utTapVWtc7/master/repro/patches'
for fn in ['configuration_regresslm.py','modeling_regresslm.py']:
    urllib.request.urlretrieve(f'{PATCH_RAW}/{fn}', os.path.join(CKPT_DIR, fn))
# Clear any cached transformers_modules copy so HF re-imports the PATCHED files.
for d in glob.glob(os.path.expanduser('~/.cache/huggingface/modules/transformers_modules/*')):
    if os.path.exists(os.path.join(d,'configuration_regresslm.py')):
        shutil.rmtree(d, ignore_errors=True)

tok = AutoTokenizer.from_pretrained(CKPT_DIR, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CKPT_DIR, trust_remote_code=True, torch_dtype=dtype).to(device).eval()
N_OUT = int(getattr(model.config,'num_tokens_per_obj',8)) * int(getattr(model.config,'max_num_objs',1))
print(f'params={sum(p.numel() for p in model.parameters())/1e6:.1f}M | N_OUT={N_OUT} | loaded+patched from {CKPT_DIR}')

In [ ]:
from huggingface_hub import hf_hub_download
# Colab's pyarrow doesn't resolve hf:// URIs -> download the parquet locally once (5.6 GB, cached).
PARQUET_LOCAL = hf_hub_download('akhauriyash/Code-Regression', 'data.parquet', repo_type='dataset')
DATASET = ds.dataset(PARQUET_LOCAL, format='parquet')

def fetch(space, lang=None, limit=512):
    flt = pa.compute.equal(pa.compute.field('space'), space)
    ins, tgts = [], []
    for b in DATASET.scanner(columns=['input','target','metadata'], filter=flt, batch_size=2048).to_batches():
        for inp, tgt, md in zip(b.column('input'), b.column('target'), b.column('metadata')):
            try: mdd = literal_eval(md.as_py()) if md.is_valid else {}
            except Exception: mdd = {}
            if lang is not None and mdd.get('language') != lang: continue
            if not tgt.is_valid or not inp.is_valid: continue
            x = inp.as_py()
            x = f"# {space}\n# Language: {mdd.get('language')}\n{x}" if space=='CDSS' else f"{space}\n{x}"
            ins.append(x); tgts.append(float(tgt.as_py()))
            if len(ins) >= limit: return ins, tgts
    return ins, tgts

@torch.inference_mode()
def predict(ins, num_samples=8, batch_size=16):
    preds=[math.nan]*len(ins)
    for i in range(0,len(ins),batch_size):
        chunk=ins[i:i+batch_size]
        enc=tok(chunk, return_tensors='pt', truncation=True, padding=True, max_length=2048).to(device)
        draws=[]
        for _ in range(num_samples):
            out=model.generate(**enc, do_sample=True, top_p=0.95, temperature=1.0,
                               min_new_tokens=N_OUT, max_new_tokens=N_OUT,
                               pad_token_id=getattr(tok,'pad_token_id',0))
            seqs=out.view(enc['input_ids'].shape[0], -1)
            vals=[]
            for r in range(seqs.shape[0]):
                try:
                    f=tok.token_ids_to_floats(seqs[r].tolist()); vals.append(float(f[0] if isinstance(f,(list,tuple)) else f))
                except Exception: vals.append(math.nan)
            draws.append(vals)
        med=np.nanmedian(np.array(draws,dtype=float),axis=0)
        for j,v in enumerate(med): preds[i+j]=float(v)
        print(f'  batch {i//batch_size+1}: {i+len(chunk)}/{len(ins)}', flush=True)
    return preds

def spear(ins, tgts):
    yp=np.array(predict(ins),dtype=float); yt=np.array(tgts,dtype=float)
    v=np.isfinite(yt)&np.isfinite(yp)
    return float(stats.spearmanr(yt[v],yp[v]).correlation), int(v.sum())

## Claim 2 — APPS Spearman (target > 0.9; reference 0.926)

In [ ]:
ins, tgts = fetch('APPS', limit=512)
appts_rho, appts_n = spear(ins, tgts)
print(f'APPS Spearman = {appts_rho:.3f}  (n={appts_n})  | claim: >0.9 | ref 0.926')

In [ ]:
ins, tgts = fetch('KBSS', limit=512)
kbss_rho, kbss_n = spear(ins, tgts)
print(f'KBSS Spearman = {kbss_rho:.3f}  (n={kbss_n})  | ref 0.527')

## Claim 3 — CodeNet languages (target: >0.5 average across 17 languages)

In [ ]:
# Enumerate CDSS languages (metadata column only -> cheap)
from collections import Counter
flt = pa.compute.equal(pa.compute.field('space'), 'CDSS')
ctr = Counter(); scanned=0
for b in DATASET.scanner(columns=['metadata'], filter=flt, batch_size=65536).to_batches():
    for md in b.column('metadata'):
        try: d=literal_eval(md.as_py())
        except Exception: d={}
        ctr[d.get('language','<none>')]+=1; scanned+=1
    if scanned >= 300000: break
langs=[l for l,_ in ctr.most_common(17)]
print(f'CDSS top languages (scanned {scanned}):', langs)

In [ ]:
per_lang={}
for l in langs:
    ins, tgts = fetch('CDSS', lang=l, limit=200)
    if len(ins) < 20: 
        print(f'  {l}: skip (only {len(ins)} rows)'); continue
    rho, n = spear(ins, tgts)
    per_lang[l]=(rho,n)
    print(f'  {l:14s} Spearman={rho:.3f}  (n={n})')
avg=np.nanmean([r for r,_ in per_lang.values()])
print(f'\nCDSS average Spearman across {len(per_lang)} langs = {avg:.3f}  | claim: >0.5')

## Summary vs Table 3

In [ ]:
import pandas as pd
rows=[['KBSS',kbss_rho,kbss_n,'0.527'], ['CDSS-avg',avg,len(per_lang),'0.787 (overall)'], ['APPS',appts_rho,appts_n,'0.926']]
df=pd.DataFrame(rows, columns=['space','spearman_repro','n','table3_ref']).set_index('space')
df['claim']=['-','>0.5 avg','>0.9']
df.to_csv('regresslm_table3_results.csv')
df